# imports

In [ ]:
# install required packages (run once)
!pip install wordcloud geopandas==0.14.4 -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter
from wordcloud import WordCloud
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

# Part A - Cleaning & Sentiment Overview

## Data Cleaning

The dataset contains 2000 climate-related tweets. Before visualizing, we need to:
- Parse dates properly
- Detect and handle duplicates
- Handle missing values in like_count and retweet_count

### A1 - Load and Inspect

In [ ]:
# load the dataset
df = pd.read_csv('Climate_Tweets_Sample.csv')

# initial inspection
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### A2 - Date Parsing

Convert the date column from string to datetime for proper time-series operations.

In [ ]:
# check current date format
print(f"Current dtype: {df['date'].dtype}")
print(f"Sample values: {df['date'].head(5).tolist()}")

# convert to datetime
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')

# verify conversion
print(f"\nNew dtype: {df['date'].dtype}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

### A3 - Duplicate Detection and Handling

Check for both full-row duplicates and text-level duplicates.

In [ ]:
# check for exact row duplicates
print(f"Exact row duplicates: {df.duplicated().sum()}")
print(f"Duplicate tweet_ids: {df.duplicated(subset=['tweet_id']).sum()}")

# check for duplicate tweet text
text_dupes = df[df.duplicated(subset=['text'], keep=False)].sort_values('text')
print(f"\nRows with duplicate text: {len(text_dupes)}")
print(f"Unique duplicate texts: {text_dupes['text'].nunique()}")

# show example
print("\nExample duplicate text group:")
sample_text = text_dupes['text'].value_counts().index[0]
print(df[df['text'] == sample_text][['tweet_id', 'date', 'country', 'sentiment', 'text']])

In [ ]:
# drop text-level duplicates, keeping the first occurrence
before = len(df)
df = df.drop_duplicates(subset=['text'], keep='first')
after = len(df)
print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Removed: {before - after} duplicate text entries")

**Justification:** 160 rows share identical tweet text with different tweet_ids, countries, and dates. These are likely reposted or templated tweets. Since the text content is identical, keeping all copies would inflate word frequency counts and sentiment distributions. I keep the first occurrence of each unique text and drop the rest.

### A4 - Handle Missing Values

Check and handle missing values in like_count and retweet_count.

In [ ]:
# check missing values after duplicate removal
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")
print(f"Missing retweet_count: {df['retweet_count'].isna().sum()} ({df['retweet_count'].isna().mean()*100:.1f}%)")
print(f"Missing like_count: {df['like_count'].isna().sum()} ({df['like_count'].isna().mean()*100:.1f}%)")

In [ ]:
# impute missing values with median
df['retweet_count'] = df['retweet_count'].fillna(df['retweet_count'].median())
df['like_count'] = df['like_count'].fillna(df['like_count'].median())

# cast to int since counts are whole numbers
df['retweet_count'] = df['retweet_count'].astype(int)
df['like_count'] = df['like_count'].astype(int)

# verify
print("Missing values after imputation:")
print(df.isnull().sum())
print(f"\nData types:")
print(df.dtypes)

**Justification:** Missing values in retweet_count (33 rows, ~1.8%) and like_count (30 rows, ~1.6%) represent a small fraction of the dataset. I chose median imputation because it preserves all rows for downstream analysis (sentiment counts, word clouds, etc.), the median is robust to the right-skewed distribution of engagement metrics (retweet max = 330, like max = 867), and the missing proportion is small enough that imputation won't meaningfully distort the data.

### Cleaning Summary

In [ ]:
print("=== CLEANING SUMMARY ===")
print(f"Final shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Sentiments: {df['sentiment'].value_counts().to_dict()}")
print(f"Countries: {df['country'].nunique()}")
print(f"Keywords: {df['keyword'].nunique()}")

# save cleaned dataset
df.to_csv('Climate_Tweets_Cleaned.csv', index=False)
print("\nCleaned dataset saved.")

---
## Sentiment Visualization

### A5 - Overall Sentiment Distribution

Bar chart showing the distribution of tweet sentiments across all countries.

In [ ]:
# count sentiments
sentiment_counts = df['sentiment'].value_counts().sort_index()
print(sentiment_counts)

In [ ]:
# bar chart: overall sentiment distribution
# Okabe-Ito palette, colourblind friendly
sentiment_colors = {
    'negative': '#D55E00',
    'neutral': '#56B4E9',
    'positive': '#009E73'
}

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(sentiment_counts.index,
              sentiment_counts.values,
              color=[sentiment_colors[s] for s in sentiment_counts.index],
              edgecolor='white',
              width=0.6)

# value labels on top of bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 8,
            f'{int(height)} ({height/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=11)

ax.set_xlabel('Sentiment', fontsize=12)
ax.set_ylabel('Number of Tweets', fontsize=12)
ax.set_title('Overall Distribution of Tweet Sentiments', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(axis='both', labelsize=11)
ax.set_ylim(0, sentiment_counts.max() * 1.15)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

ax.yaxis.grid(True, linestyle='-', alpha=0.15)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### A6 - Sentiment Distribution by Top 10 Countries

Identify the top 10 countries with the most tweets, then visualize sentiment breakdown.

In [ ]:
# identify top 10 countries by tweet count
top10_countries = df['country'].value_counts().head(10)
print("Top 10 countries by tweet count:")
print(top10_countries)

In [ ]:
# filter to top 10 countries and build sentiment crosstab
df_top10 = df[df['country'].isin(top10_countries.index)]
sentiment_by_country = pd.crosstab(df_top10['country'], df_top10['sentiment'])

# sort by total tweets descending
sentiment_by_country = sentiment_by_country.loc[top10_countries.index]

print(sentiment_by_country)

In [ ]:
# grouped bar chart: sentiment distribution across top 10 countries
# Okabe-Ito palette, colourblind friendly
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(sentiment_by_country))
width = 0.25

for i, (sentiment, color) in enumerate(sentiment_colors.items()):
    if sentiment in sentiment_by_country.columns:
        values = sentiment_by_country[sentiment].values
        bars = ax.bar(x + i * width, values, width,
                      label=sentiment.capitalize(), color=color, edgecolor='white')

ax.set_xlabel('Country', fontsize=12)
ax.set_ylabel('Number of Tweets', fontsize=12)
ax.set_title('Sentiment Distribution Across Top 10 Countries by Tweet Volume',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x + width)
ax.set_xticklabels(sentiment_by_country.index, rotation=35, ha='right', fontsize=10)
ax.legend(frameon=False, fontsize=11)
ax.tick_params(axis='y', labelsize=11)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

ax.yaxis.grid(True, linestyle='-', alpha=0.15)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

# Part B - Text Preprocessing & Word Cloud

## Text Preprocessing

Steps:
1. Lowercase all text
2. Remove punctuation and numbers
3. Remove stopwords (English common words + domain-specific template filler)
4. Filter tokens shorter than 3 characters

In [ ]:
# define stopwords manually (NLTK unavailable in this environment)
STOPWORDS = set("""i me my myself we our ours ourselves you your yours yourself yourselves
he him his himself she her hers herself it its itself they them their theirs themselves
what which who whom this that these those am is are was were be been being have has had
having do does did doing a an the and but if or because as until while of at by for with
about against between through during before after above below to from up down in out on
off over under again further then once here there when where why how all both each few
more most other some such no nor not only own same so than too very s t can will just don
should now d ll m o re ve y ain aren couldn didn doesn hadn hasn haven isn ma mightn mustn
needn shan shouldn wasn weren won wouldn could would also much get""".split())

# domain-specific stopwords: template filler words that appear across all sentiments
# without carrying meaningful climate content
STOPWORDS.update(['check', 'figures', 'details', 'incoming', 'source', 'comments',
                  'data', 'attached', 'learn', 'soon', 'share', 'responsibly',
                  'stay', 'informed', 'think', 'report', 'update', 'opinion',
                  'breaking', 'reminder', 'heads', 'fyi', 'research', 'new',
                  'overview', 'trends', 'latest', 'face', 'discussion', 'continues',
                  'released', 'stats'])

def preprocess_text(text):
    text = text.lower()                          # lowercase
    text = re.sub(r'[^\w\s]', '', text)          # remove punctuation
    text = re.sub(r'\d+', '', text)               # remove numbers
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens

# apply preprocessing
df['tokens'] = df['text'].apply(preprocess_text)

# check results
all_tokens = [t for tokens in df['tokens'] for t in tokens]
print(f"Total tokens after preprocessing: {len(all_tokens)}")
print(f"Unique tokens: {len(set(all_tokens))}")
print(f"\nSample preprocessed tweet:")
print(f"  Original: {df['text'].iloc[0]}")
print(f"  Tokens:   {df['tokens'].iloc[0]}")

**Preprocessing choices:**
- **Lowercasing** standardizes case for frequency counting.
- **Punctuation removal** strips noise characters (colons, semicolons, periods) common in the tweet format.
- **Stopwords** combine a standard English stopword list with domain-specific filler words ("Breaking:", "Report:", "FYI:", "details incoming", "overview", "trends", etc.) that appear in the templated tweet structure and don't carry topical meaning.
- **Minimum token length of 3** removes residual two-letter fragments.
- I opted not to lemmatize because the key terms in this dataset are mostly base-form nouns and adjectives (emissions, drought, warming) that don't benefit from stemming.

## Keyword & Word Cloud Analysis

### B1 - Top 20 Most Frequent Terms

Identify and visualize the 20 most common words across all tweets.

In [ ]:
# top 20 most frequent terms
top20 = Counter(all_tokens).most_common(20)
terms, counts = zip(*top20)

print("Top 20 terms:")
for term, count in top20:
    print(f"  {term:20s} {count}")

In [ ]:
# bar chart: top 20 most frequent terms
fig, ax = plt.subplots(figsize=(10, 7))

colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(terms)))[::-1]
bars = ax.barh(list(reversed(terms)), list(reversed(counts)), color=colors)

# value labels
for bar in bars:
    width = bar.get_width()
    ax.text(width + 3, bar.get_y() + bar.get_height()/2,
            f'{int(width)}', ha='left', va='center', fontsize=10)

ax.set_xlabel('Frequency', fontsize=12)
ax.set_title('Top 20 Most Frequent Terms in Climate Tweets', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(axis='both', labelsize=11)

for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)

ax.xaxis.grid(True, linestyle='-', alpha=0.15)
ax.set_axisbelow(True)
ax.set_xlim(0, max(counts) * 1.15)

plt.tight_layout()
plt.show()

### B2 - Word Clouds by Sentiment

Separate word clouds for positive and negative tweets to compare language patterns.

In [ ]:
# separate tokens by sentiment
pos_tokens = [t for tokens, s in zip(df['tokens'], df['sentiment']) if s == 'positive' for t in tokens]
neg_tokens = [t for tokens, s in zip(df['tokens'], df['sentiment']) if s == 'negative' for t in tokens]

print(f"Positive tokens: {len(pos_tokens)}")
print(f"Negative tokens: {len(neg_tokens)}")
print(f"\nTop 10 positive: {Counter(pos_tokens).most_common(10)}")
print(f"Top 10 negative: {Counter(neg_tokens).most_common(10)}")

In [ ]:
# word cloud: positive tweets
wc_pos = WordCloud(width=800, height=400, background_color='white',
                   colormap='Greens', max_words=80,
                   collocations=False,
                   prefer_horizontal=0.7).generate(' '.join(pos_tokens))

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(wc_pos, interpolation='bilinear')
ax.set_title('Word Cloud: Positive Tweets', fontsize=14, fontweight='bold', pad=10, color='#009E73')
ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# word cloud: negative tweets
wc_neg = WordCloud(width=800, height=400, background_color='white',
                   colormap='Oranges', max_words=80,
                   collocations=False,
                   prefer_horizontal=0.7).generate(' '.join(neg_tokens))

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(wc_neg, interpolation='bilinear')
ax.set_title('Word Cloud: Negative Tweets', fontsize=14, fontweight='bold', pad=10, color='#D55E00')
ax.axis('off')

plt.tight_layout()
plt.show()

**Observations:** The positive word cloud is dominated by terms like "encouraging", "signs", "progress", "promising", and "boost", reflecting optimistic framing around policies and sustainability. The negative cloud emphasizes "concerns", "worsening", "impacts", "setbacks", and "alarm", consistent with pessimistic coverage of environmental decline. Both clouds share core topic words (emissions, climate, energy) but differ sharply in their framing verbs and adjectives.

# Part C - Geospatial Visualization

Choropleth map showing the number of climate tweets per country.

### Country Name Standardization

In [ ]:
# count tweets per country
country_counts = df['country'].value_counts().reset_index()
country_counts.columns = ['country', 'tweet_count']

# country name mapping for Natural Earth shapefile
name_map = {
    'United States': 'United States of America',
}

country_counts['map_name'] = country_counts['country'].map(lambda x: name_map.get(x, x))

# load world shapefile (built into geopandas)
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))

# check for unmatched countries
unmatched = [c for c in country_counts['map_name'] if c not in world['name'].values]
matched = len(country_counts) - len(unmatched)
print(f"Matched: {matched} / {len(country_counts)}")
print(f"Unmatched: {unmatched}")

# merge
merged = world.merge(country_counts, left_on='name', right_on='map_name', how='left')

**Country name handling:** The only inconsistency is "United States" vs "United States of America" in the shapefile. Singapore is a city-state too small to render at this map resolution and doesn't appear in the Natural Earth low-res dataset, so it's excluded (1 country out of 39).

### Choropleth Map

In [ ]:
# choropleth: tweet count by country
fig, ax = plt.subplots(figsize=(14, 7))

# countries without data in light gray
merged[merged['tweet_count'].isna()].plot(ax=ax, color='#E0E0E0', edgecolor='white', linewidth=0.5)

# countries with data
merged_data = merged[merged['tweet_count'].notna()]
merged_data.plot(column='tweet_count', ax=ax, legend=True,
                 cmap='Blues', edgecolor='white', linewidth=0.5,
                 legend_kwds={'label': 'Tweet Count',
                              'orientation': 'vertical',
                              'shrink': 0.6,
                              'pad': 0.02})

ax.set_title('Number of Climate Tweets by Country', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(-170, 180)
ax.set_ylim(-60, 85)
ax.axis('off')

# note about Singapore
ax.text(-165, -55, 'Note: Singapore not visible at this map resolution.',
        fontsize=8, color='gray', style='italic')

plt.tight_layout()
plt.show()

# Part D - Uncertainty & Association

## D1 - Monthly Negative Sentiment Rate + 95% Confidence Interval

Compute the monthly proportion of negative tweets and visualize with Wilson score confidence intervals.

In [ ]:
# compute monthly negative sentiment rate
df['month'] = df['date'].dt.to_period('M')
df['is_negative'] = (df['sentiment'] == 'negative').astype(int)

monthly = df.groupby('month').agg(
    n_negative=('is_negative', 'sum'),
    n_total=('is_negative', 'count')
).reset_index()

monthly['rate'] = monthly['n_negative'] / monthly['n_total']

# Wilson score confidence interval
z = 1.96  # 95% CI
n = monthly['n_total']
p = monthly['rate']

monthly['ci_low'] = (p + z**2/(2*n) - z * np.sqrt((p*(1-p) + z**2/(4*n))/n)) / (1 + z**2/n)
monthly['ci_high'] = (p + z**2/(2*n) + z * np.sqrt((p*(1-p) + z**2/(4*n))/n)) / (1 + z**2/n)

monthly['month_dt'] = monthly['month'].dt.to_timestamp()

# check results
print(monthly[['month', 'n_negative', 'n_total', 'rate', 'ci_low', 'ci_high']].head(12).to_string())

**Method:** I use the Wilson score interval rather than the normal approximation because it provides better coverage for proportions near 0 or 1 and for months with smaller sample sizes. The Wilson interval accounts for the bounded nature of proportions and never produces impossible values below 0 or above 1. At z = 1.96 (95% confidence), each monthly point has a CI that reflects both the estimated proportion and the sample size for that month.

In [ ]:
# time-series: monthly negative sentiment rate with 95% CI
fig, ax = plt.subplots(figsize=(14, 6))

# confidence band
ax.fill_between(monthly['month_dt'], monthly['ci_low'], monthly['ci_high'],
                alpha=0.2, color='#D55E00', label='95% Wilson CI')

# rate line
ax.plot(monthly['month_dt'], monthly['rate'],
        color='#D55E00', linewidth=2, marker='o', markersize=4, label='Negative Rate')

# overall mean reference line
overall_rate = df['is_negative'].mean()
ax.axhline(y=overall_rate, color='gray', linestyle='--', linewidth=1, alpha=0.6,
           label=f'Overall Mean ({overall_rate:.1%})')

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Negative Tweet Rate', fontsize=12)
ax.set_title('Monthly Negative Sentiment Rate with 95% Confidence Interval',
             fontsize=14, fontweight='bold', pad=15)
ax.legend(frameon=False, fontsize=10, loc='upper right')
ax.tick_params(axis='both', labelsize=10)

# format axes
ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

ax.yaxis.grid(True, linestyle='-', alpha=0.15)
ax.set_axisbelow(True)
ax.set_ylim(0, 0.75)

plt.tight_layout()
plt.show()

The plot shows that negative sentiment fluctuates between roughly 18% and 52% month to month. The CI bands are wider in months with fewer tweets and narrower in high-volume months, which is expected. There is no obvious long-term upward or downward trend in negative sentiment over the 2021 to 2023 window, though individual months show notable spikes (e.g. July 2021 at ~48%, June 2022 at ~52%).

## D2 - Scatter/Bubble: Likes vs Retweets

Select the top 5 countries by tweet volume to avoid an overly dense plot.

In [ ]:
# select top 5 countries by tweet count
top5 = df['country'].value_counts().head(5).index.tolist()
df_scatter = df[df['country'].isin(top5)].copy()

print(f"Top 5 countries: {top5}")
print(f"Rows for scatter: {len(df_scatter)}")
print(f"\nSentiment breakdown in subset:")
print(df_scatter['sentiment'].value_counts())

In [ ]:
# scatter: likes vs retweets, coloured by sentiment
fig, ax = plt.subplots(figsize=(10, 7))

for sentiment, color in sentiment_colors.items():
    subset = df_scatter[df_scatter['sentiment'] == sentiment]
    ax.scatter(subset['retweet_count'], subset['like_count'],
               c=color, label=sentiment.capitalize(), alpha=0.6, s=40,
               edgecolors='white', linewidths=0.5)

ax.set_xlabel('Retweet Count', fontsize=12)
ax.set_ylabel('Like Count', fontsize=12)
ax.set_title('Likes vs Retweets for Top 5 Countries by Tweet Volume\nColoured by Sentiment',
             fontsize=14, fontweight='bold', pad=15)
ax.legend(title='Sentiment', frameon=False, fontsize=10, title_fontsize=11)
ax.tick_params(axis='both', labelsize=11)

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

ax.xaxis.grid(True, linestyle='-', alpha=0.15)
ax.yaxis.grid(True, linestyle='-', alpha=0.15)
ax.set_axisbelow(True)

# sample size
ax.text(0.02, 0.98, f'n = {len(df_scatter)}', transform=ax.transAxes,
        fontsize=10, va='top', color='gray')

plt.tight_layout()
plt.show()

**Discussion:** The scatter plot shows a broadly positive association between retweet count and like count, which is expected since both measure engagement. Sentiment doesn't cluster in any particular region of the engagement space: negative, neutral, and positive tweets are all scattered across the full range of likes and retweets. This suggests that engagement levels are driven more by topic or timing than by whether the tweet is framed positively or negatively. There is no evidence that negative tweets attract disproportionately more engagement than positive ones in this subset.

# Part E - Design & Insight

## E1 - Design Choices

**Colour Scales** All visualizations use the Okabe-Ito palette, designed for colourblind accessibility (I am colourblind). Sentiment is consistently encoded across all charts: red-orange (#D55E00) for negative, sky blue (#56B4E9) for neutral, and green (#009E73) for positive. The choropleth uses a sequential Blues scale where darker shading corresponds to higher tweet counts, creating an intuitive "more = darker" mapping. Word clouds use green (positive) and orange-red (negative) colourmaps to reinforce the sentiment association.

**Uncertainty** The Wilson score interval was chosen over the normal approximation for the monthly negative rate because it handles small-sample months better and never produces CI bounds outside [0, 1]. The shaded band encodes uncertainty continuously rather than using error bars, reducing visual clutter and making temporal trends easier to follow.

**Cognitive Load** Horizontal bars eliminate rotated text labels. Chart junk (borders, unnecessary gridlines, heavy spines) is removed throughout. Value labels are placed directly beside or above bars to reduce eye movement. The scatter plot uses transparency (alpha=0.6) and white edge markers to manage overplotting in dense regions.

**Contiguity** Labels are placed adjacent to their data points across all charts. The CI band sits directly on the rate line. Legend placement is consistent and near the relevant data region.

## E2 - Key Insights

Negative sentiment is the most prevalent category (36.3%), slightly outpacing positive (33.0%) and neutral (30.7%), suggesting climate discourse on social media leans pessimistic overall. This distribution is not uniform across countries: Mexico and Saudi Arabia show substantially higher negative proportions, while Italy has the strongest positive skew among the top 10 countries. The word frequency analysis confirms that the positive-negative split is driven by distinct framing language: positive tweets cluster around "encouraging", "progress", and "boost", while negative tweets emphasize "concerns", "worsening", and "setbacks", even when discussing the same underlying topics. The monthly CI plot shows no clear temporal trend in negative sentiment over the three-year window, which suggests that the overall pessimistic tilt is structural rather than driven by specific events. Finally, the engagement analysis reveals that sentiment does not predict virality: likes and retweets are distributed similarly across all three sentiment categories, implying that content topic and timing matter more than emotional framing for driving engagement.